# FreqNet Inference Setup
This notebook will download your dataset from Google Drive, unpack it, load the `FreqNet` model weights, run inference, and generate detailed metrics.

In [ ]:
# 1. Install required libraries
!pip install gdown scikit-learn seaborn matplotlib torch torchvision

In [ ]:
# 2. Imports and Path setup
import sys
import os
import zipfile
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import numpy as np
import gdown

# Add the FreqNet repo to path to import model
sys.path.append(r"C:\hfvt\FreqNet-DeepfakeDetection")
from networks.freqnet import freqnet

In [ ]:
# 3. Google Drive Download
# ---------------------------------------------------------
# IMPORTANT: Replace the ID below with your Google Drive File ID.
# It should be a public link for `gdown` to work directly.
# If private, you can manually upload the zip file to this directory.
# ---------------------------------------------------------
DRIVE_FILE_ID = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE"
ZIP_PATH = "dataset.zip"
EXTRACT_DIR = "dataset_extracted"

if not os.path.exists(ZIP_PATH):
    print("Downloading dataset from Google Drive...")
    url = f'https://drive.google.com/uc?id={DRIVE_FILE_ID}'
    gdown.download(url, ZIP_PATH, quiet=False)
else:
    print("dataset.zip already exists. Skipping download.")

if not os.path.exists(EXTRACT_DIR):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

In [ ]:
# 4. Data Loading
# ---------------------------------------------------------
# Update data_dir if the zip extracts a parent folder first 
# (e.g., EXTRACT_DIR + "/my_dataset").
data_dir = EXTRACT_DIR 

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

try:
    dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    # Small batch size since dataset is 140 images
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=2)
    print(f"Loaded {len(dataset)} images.")
    print(f"Classes detected: {dataset.class_to_idx}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure your dataset folder has subfolders for each class (e.g., 'real' and 'fake').")

In [ ]:
# 5. Model Loading
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = freqnet()
weights_path = r"C:\hfvt\FreqNet-DeepfakeDetection\4-classes-freqnet-v2.pth"

# Load weights
checkpoint = torch.load(weights_path, map_location=device)
if 'state_dict' in checkpoint:
    model.load_state_dict(checkpoint['state_dict'])
else:
    model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()
print("FreqNet model loaded successfully.")

In [ ]:
# 6. Inference Loop
# ---------------------------------------------------------
all_preds = []
all_probs = []
all_labels = []

# Safely map ImageFolder classes to FreqNet classes (fake=1, real=0)
class_names = [c.lower() for c in dataset.classes]
fake_idx = class_names.index('fake') if 'fake' in class_names else 1
real_idx = class_names.index('real') if 'real' in class_names else 0

print("Running inference...")
with torch.no_grad():
    for images, labels in dataloader:
        images = images.to(device)
        
        # Convert dataset labels to 0 (real), 1 (fake)
        true_labels = []
        for l in labels:
            if l.item() == fake_idx:
                true_labels.append(1)
            else:
                true_labels.append(0)
        true_labels = torch.tensor(true_labels).to(device)
        
        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()
        
        # Ensure it works for batch size 1
        if probs.dim() == 0:
            probs = probs.unsqueeze(0)
            preds = preds.unsqueeze(0)
            
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(true_labels.cpu().numpy())
        
print("Inference complete.")

In [ ]:
# 7. Metrics Evaluation
# ---------------------------------------------------------
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

try:
    roc_auc = roc_auc_score(all_labels, all_probs)
except ValueError:
    roc_auc = float('nan')
    print("ROC AUC cannot be calculated (e.g. only one class present in data)")

print("=== FreqNet Inference Results ===")
print(f"Total Images: {len(all_labels)}")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
if not np.isnan(roc_auc):
    print(f"ROC-AUC:   {roc_auc:.4f}")
print("=================================")

In [ ]:
# 8. Visualizations: Confusion Matrix & ROC Curve
# ---------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, 
            xticklabels=['Real (0)', 'Fake (1)'], yticklabels=['Real (0)', 'Fake (1)'])
ax1.set_title("Confusion Matrix")
ax1.set_ylabel("Actual")
ax1.set_xlabel("Predicted")

# ROC Curve
if not np.isnan(roc_auc):
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax2.set_xlim([0.0, 1.0])
    ax2.set_ylim([0.0, 1.05])
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate')
    ax2.set_title('Receiver Operating Characteristic')
    ax2.legend(loc="lower right")
else:
    ax2.text(0.5, 0.5, 'ROC AUC Unavailable', ha='center', va='center')

plt.tight_layout()
plt.show()